In [ ]:
import secrets

# ======== CHANGE THIS PER ACCOUNT: replica-a / replica-b / replica-c / replica-d ========
REPLICA_TAG = "replica-a"
# =========================================================================================

MODEL  = "Qwen/Qwen2.5-3B-Instruct"
BRANCH = "claude/friendly-turing-v9y7vj"
KV_BLOCKS = 375                      # 375 blocks * 16 tokens = 6,000 tokens of cache (small on purpose)
VLLM_PORT, SIDECAR_PORT = 8000, 9100
ADMIN_TOKEN = secrets.token_hex(8)   # protects the sidecar's /chaos endpoints; the router does not need it

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

print(f"\nREPLICA_TAG = {REPLICA_TAG}")
print(f"MODEL       = {MODEL}")
print(f"KV_BLOCKS   = {KV_BLOCKS}")
print(f"ADMIN_TOKEN = {ADMIN_TOKEN}   (keep private; the router does not need it)")

In [ ]:
import os
os.chdir("/content")

# Clone, or update an existing clone to the latest commit of BRANCH.
if os.path.isdir("distrouter/.git"):
    !cd distrouter && git fetch --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} https://github.com/Akshataram/distrouter.git

os.chdir("/content/distrouter")
!git log --oneline -1

!pip install -q vllm
!pip install -q -r requirements.txt
!pip uninstall -y -q torchaudio || true

# The sidecar needs these four; fail loudly here instead of later.
import importlib
for m in ("fastapi", "uvicorn", "httpx", "prometheus_client"):
    importlib.import_module(m)
print("setup done: repo at", os.getcwd(), "| sidecar dependencies import OK")

In [ ]:
import os, time, subprocess, urllib.request
os.chdir("/content/distrouter")

subprocess.run(["pkill", "-9", "-f", "replica_sidecar"])
subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints"])
time.sleep(8)
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

LOG_DIR = "logs"; os.makedirs(LOG_DIR, exist_ok=True)
vllm_log_path = f"{LOG_DIR}/vllm.log"
vllm_log = open(vllm_log_path, "w")

# max-model-len MUST be <= KV_BLOCKS*16, or vLLM aborts at startup (375*16 = 6000 >= 4096).
MAX_MODEL_LEN = 4096

vllm = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", str(VLLM_PORT),
     "--enable-prefix-caching", "--enable-prompt-tokens-details",
     "--block-size", "16", "--max-num-seqs", "32",
     "--max-model-len", str(MAX_MODEL_LEN),
     "--dtype", "half",
     "--num-gpu-blocks-override", str(KV_BLOCKS),
     "--gpu-memory-utilization", "0.90"],
    stdout=vllm_log, stderr=subprocess.STDOUT,
)
print(f"vllm pid={vllm.pid}, log={vllm_log_path}\n")

start = time.monotonic()
while time.monotonic() - start < 1800:
    if vllm.poll() is not None:
        print(f"\n!! vLLM EXITED (code {vllm.returncode}). Last 30 lines:\n")
        !tail -30 {vllm_log_path}
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=3) as r:
            if r.status == 200:
                print(f"\n*** vLLM HEALTHY after {(time.monotonic()-start)/60:.1f} min ***")
                !grep -iE "GPU blocks|KV cache size" {vllm_log_path} | tail -3
                break
    except Exception:
        pass
    with open(vllm_log_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    print(f"[{(time.monotonic()-start)/60:5.1f}m] {lines[-1][:140] if lines else '(no output yet)'}")
    time.sleep(10)

In [ ]:
import os, subprocess, time, urllib.request

os.chdir("/content/distrouter")      # so `python -m swiftserve.replica_sidecar` can import the repo
LOG_DIR = "logs"; os.makedirs(LOG_DIR, exist_ok=True)

# 1. vLLM must already be healthy
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=5) as r:
        print(f"vLLM health: {r.status}")
except Exception as e:
    raise SystemExit(f"vLLM is NOT healthy ({e}). Run the vLLM cell and wait for 'vLLM HEALTHY' first.")

# 2. stop any old sidecar still holding the port
subprocess.run(["pkill", "-9", "-f", "replica_sidecar"])
time.sleep(2)

# 3. start the sidecar
sidecar_log = open(f"{LOG_DIR}/sidecar.log", "w")
sidecar = subprocess.Popen(
    ["python", "-m", "swiftserve.replica_sidecar",
     "--upstream-url", f"http://127.0.0.1:{VLLM_PORT}",
     "--port", str(SIDECAR_PORT),
     "--admin-token", ADMIN_TOKEN],
    stdout=sidecar_log, stderr=subprocess.STDOUT,
)

# 4. wait up to 60 s for it to answer
start = time.monotonic()
while time.monotonic() - start < 60:
    if sidecar.poll() is not None:
        print(f"!! sidecar EXITED (code {sidecar.returncode}). Its log:\n")
        print(open(f"{LOG_DIR}/sidecar.log").read()[-3000:])
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{SIDECAR_PORT}/health", timeout=3) as r:
            print(f"sidecar -> vLLM health: {r.status}  (200 = good)")
            break
    except Exception:
        time.sleep(2)
else:
    print("!! sidecar still not answering after 60 s. Its log:\n")
    print(open(f"{LOG_DIR}/sidecar.log").read()[-3000:])

In [ ]:
import json, time, re, urllib.request

def tail_log(path=f"{LOG_DIR}/vllm.log", n=600):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

found = re.findall(r"GPU (?:KV cache size|blocks)[^0-9]*([0-9,]+)", tail_log())
print(f"KV_BLOCKS requested : {KV_BLOCKS}  (= {KV_BLOCKS*16:,} tokens)")
print(f"vLLM reported       : {found if found else 'NOT FOUND'}")
print("  (if these disagree, no cache pressure -> all policies will tie)\n")

BIG_SYSTEM = "You are a meticulous customer support agent. " * 200

def call(user, max_tokens=16):
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "system", "content": BIG_SYSTEM},
                     {"role": "user", "content": user}],
        "max_tokens": max_tokens, "stream": True,
        "stream_options": {"include_usage": True},
    }).encode()
    req = urllib.request.Request(
        f"http://127.0.0.1:{SIDECAR_PORT}/v1/chat/completions", data=body,
        headers={"Content-Type": "application/json"}, method="POST")
    t0, ttft, usage = time.monotonic(), None, None
    with urllib.request.urlopen(req, timeout=180) as resp:
        for raw in resp:
            line = raw.decode("utf-8", "replace").strip()
            if not line.startswith("data:"):
                continue
            payload = line[5:].strip()
            if payload == "[DONE]":
                break
            chunk = json.loads(payload)
            ch = chunk.get("choices") or []
            if ch and (ch[0].get("delta") or {}).get("content") and ttft is None:
                ttft = (time.monotonic() - t0) * 1000
            if chunk.get("usage"):
                usage = chunk["usage"]
    return ttft, usage

print(f"{'call':<28}{'TTFT':>10}{'prompt_tok':>12}{'cached_tok':>12}")
rows = []
for label, q in [("1st (cold prefix)", "reset my password"),
                 ("2nd (same system prompt)", "where is my invoice"),
                 ("3rd (same system prompt)", "cancel my plan")]:
    ttft, usage = call(q)
    cached = (usage or {}).get("prompt_tokens_details", {}).get("cached_tokens")
    prompt = (usage or {}).get("prompt_tokens")
    rows.append((label, ttft, prompt, cached))
    print(f"{label:<28}{ttft:>8.0f}ms{str(prompt):>12}{str(cached):>12}")

warm = [r for r in rows[1:] if r[3]]
if not warm:
    print("\n!! FAIL: cached_tokens is 0/None -> --enable-prompt-tokens-details not in effect")
elif rows[0][1] and warm[0][1] and rows[0][1] / warm[0][1] > 1.5:
    print(f"\nOK: prefix caching live ({rows[0][1]/warm[0][1]:.1f}x lower TTFT warm)")
else:
    print("\n?? cached_tokens fine but TTFT flat -- lengthen BIG_SYSTEM")

In [ ]:
import os, re, subprocess, time
os.chdir("/content/distrouter")
LOG_DIR = "logs"; os.makedirs(LOG_DIR, exist_ok=True)

subprocess.run(["pkill", "-9", "-f", "cloudflared"])
time.sleep(1)
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

tunnel_log = open(f"{LOG_DIR}/cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{SIDECAR_PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url, deadline = None, time.monotonic() + 90
while time.monotonic() < deadline:
    with open(f"{LOG_DIR}/cloudflared.log") as f:
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
    if m:
        public_url = m.group(0); break
    time.sleep(2)

# Prove the tunnel reaches vLLM through the sidecar (retry: new tunnels need a few seconds).
status = "no URL yet"
if public_url:
    for _ in range(10):
        out = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "-m", "10",
                              "-A", "Mozilla/5.0", f"{public_url}/health"],
                             capture_output=True, text=True).stdout.strip()
        status = out
        if out == "200":
            break
        time.sleep(3)

print("=" * 64)
print(f"  {REPLICA_TAG}")
print(f"  URL    : {public_url}      <-- COPY THIS (no trailing slash)")
print(f"  via tunnel /health : {status}   (200 = reachable from outside)")
print("=" * 64)
if status != "200":
    print("!! Tunnel health is not 200. Do not use this URL until it is. Stop this cell and run it again.")

print("\nkeep-alive running; DO NOT stop this cell")
while True:
    time.sleep(60)